# Generate Paper Embeddings with SPECTER2

This notebook creates [SPECTER2](https://github.com/allenai/SPECTER2) paper embeddings from a CSV file. Run it in Google Colab with a free or paid GPU runtime.

**Author:** [Juan Pablo Bascur](https://jpbascur.com)  
**Source:** [github.com/jpbascur/snipets/blob/main/generate_embeddings.ipynb](https://github.com/jpbascur/snipets/blob/main/generate_embeddings.ipynb)  
**License:** MIT

**Input:** a CSV file with these columns: `id`, `title`, `abstract`.

**Output:** a CSV file with no header row. The first column is the paper id, followed by 768 embedding values.

## Workflow

### Part 1: Prepare your file

Your input CSV must have three columns: `id`, `title`, and `abstract`. If you are working with a bibliometric export (e.g., from Web of Science or Scopus), you can use [Text Similarity Maker](https://huggingface.co/spaces/juanbascur/text-similarity-maker) to convert it to the correct format. If your CSV already has the right data but uses different column names, rename them before proceeding.

### Part 2: Generate embeddings

1. If you are not already in Google Colab, go to [colab.research.google.com](https://colab.research.google.com), sign in with your Google account, and open this notebook from the source link above.
2. In the Colab menu, choose **Runtime > Change runtime type**.
3. Set **Hardware accelerator** to **T4 GPU** or another available GPU.
4. Click **Save**.
5. Run cell 1 to install dependencies and load the model.
6. Run cell 2. A file upload prompt will appear at the bottom of the cell output — scroll down to find it. Click **Choose Files** and select your input CSV.
7. When the embeddings are finished, Colab will download `embeddings.csv` automatically.

## Notes

- The first run may take a few minutes because the model is downloaded from Hugging Face.
- The printed device line shows whether the notebook is using a GPU or CPU. If it says `Using GPU`, the GPU is active, which is usually much faster.

## Troubleshooting

- If `embeddings.csv` does not download automatically, your browser may have blocked it. Look for a blocked download notification in the browser toolbar and allow it, or try using Google Chrome.
- If the output file is too large to work with, you can reduce `DECIMAL_PLACES` in cell 1 to make it smaller. Ideally, do not go below 4.
- If Colab runs out of GPU memory, lower `BATCH_SIZE` to `64` or `32` in cell 1 and run again.
- If the notebook fails and you cannot find the reason, set `INSTALL_STABLE_VERSION = True` at the top of cell 1 and run it again. Note that this will take longer to install.

In [ ]:
# Reduce this number to get a smaller output file at the cost of precision.
DECIMAL_PLACES = 7

# Lower this to 64 or 32 if Colab runs out of GPU memory.
BATCH_SIZE = 128

# Set to True if the notebook fails for unknown reasons.
INSTALL_STABLE_VERSION = False

# Model constants: do not change these.
MODEL_NAME = 'allenai/specter2_base'
ADAPTER_NAME = 'allenai/specter2'
ADAPTER_ALIAS = 'proximity'

# Install dependencies
import subprocess
if INSTALL_STABLE_VERSION:
    subprocess.run(['pip', 'install', '-q', 'adapters==1.3.0', 'transformers==4.57.6'], check=True)
else:
    subprocess.run(['pip', 'install', '-q', 'adapters', 'transformers'], check=True)

# Imports
import io
import numpy as np
import pandas as pd
import torch
from adapters import AutoAdapterModel
from transformers import AutoTokenizer

# Device check
if torch.cuda.is_available():
    device = 'cuda'
    print(f'Using GPU: {torch.cuda.get_device_name(0)}')
else:
    device = 'cpu'
    print('Using CPU. This will work, but it may be slow for large CSV files.')

# Load model
print('Loading SPECTER2 model. The first run may take a few minutes...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoAdapterModel.from_pretrained(MODEL_NAME)
model.load_adapter(ADAPTER_NAME, source='hf', load_as=ADAPTER_ALIAS)
model.set_active_adapters(ADAPTER_ALIAS)
model.to(device)
model.eval()
embedding_size = model.config.hidden_size
print(f'Model ready. Embedding size: {embedding_size}')

In [ ]:
# Load input CSV
from google.colab import files
uploaded = files.upload()
if not uploaded:
    raise ValueError('No file was uploaded.')
if len(uploaded) > 1:
    raise ValueError(f'Upload one file at a time. You uploaded {len(uploaded)} files.')
filename = next(iter(uploaded))
print(f'Uploaded file: {filename}')
df = pd.read_csv(io.BytesIO(uploaded[filename]))

# Validate CSV shape
required_columns = ['id', 'title', 'abstract']
duplicate_columns = df.columns[df.columns.duplicated()].tolist()
if duplicate_columns:
    raise ValueError(f'Duplicate column names found: {duplicate_columns}')

missing_columns = [column for column in required_columns if column not in df.columns]
if missing_columns:
    raise ValueError(f'Missing required columns: {missing_columns}')

if df.empty:
    raise ValueError('The input CSV has no rows.')

print(f'Loaded {len(df)} papers.')

# Encode papers
n = len(df)
embeddings = np.zeros((n, embedding_size), dtype=np.float32)

for start in range(0, n, BATCH_SIZE):
    end = min(start + BATCH_SIZE, n)
    batch = df.iloc[start:end]
    texts = (
        batch['title'].fillna('').astype(str)
        + ' [SEP] '
        + batch['abstract'].fillna('').astype(str)
    ).tolist()

    inputs = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=512,
        return_tensors='pt',
    )
    inputs = {key: value.to(device) for key, value in inputs.items()}

    with torch.no_grad():
        output = model(**inputs)

    embeddings[start:end] = output.last_hidden_state[:, 0, :].detach().cpu().numpy()
    print(f'Encoded {end} / {n}')

# Save and download output CSV
ids = df['id'].astype(str).reset_index(drop=True)
output_df = pd.concat([ids, pd.DataFrame(embeddings)], axis=1)
output_df.to_csv('embeddings.csv', index=False, header=False, float_format=f'%.{DECIMAL_PLACES}f')

print('Saved embeddings to embeddings.csv')
files.download('embeddings.csv')